Notebook 03 - Construção do Índice FAISS da Base Maicon

Este notebook realiza a construção do índice vetorial FAISS a partir dos embeddings e metadados gerados no Notebook 02.

Os embeddings representam semanticamente os chunks da Base Maicon. A entrada utilizada para a geração de cada embedding segue a estratégia definida no Notebook 02: **`cleaned_summary → summary → text`**. O campo `embedding_source` preserva a origem textual utilizada em cada registro, permitindo a rastreabilidade dos embeddings ao longo do pipeline.

O índice vetorial é construído utilizando **FAISS IndexFlatIP**. Como os embeddings foram previamente normalizados, o produto interno corresponde à similaridade cosseno, permitindo a recuperação dos chunks semanticamente mais próximos de uma consulta.

Ao final da execução, o índice FAISS e seus respectivos artefatos são salvos para utilização no Notebook 04, responsável pela recuperação semântica e pela consulta ao modelo de linguagem no pipeline RAG.

Funções Auxiliares

In [ ]:
# Define funções auxiliares para padronizar as mensagens

def print_header(title):
    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)


def print_success(message):
    print(f"\n✅ {message}")


def print_error(message):
    print(f"\n❌ {message}")


def print_warning(message):
    print(f"\n⚠️ {message}")


def print_info(label, value):
    print(f"{label:<25} {value}")

Preparação do Ambiente

In [ ]:
# Importa as bibliotecas necessárias

import json
import pickle

from datetime import datetime
from pathlib import Path

import faiss
import numpy as np

from google.colab import drive

In [ ]:
# Instala a biblioteca FAISS

!pip install -q faiss-cpu

print(
    "FAISS instalado."
)

FAISS instalado.


In [ ]:
# Verifica se a biblioteca FAISS está disponível

print_header(
    "VERIFICAÇÃO DA BIBLIOTECA FAISS"
)

try:

    faiss_version = getattr(
        faiss,
        "__version__",
        "Não informada"
    )

    print_info(
        "Biblioteca:",
        "FAISS"
    )

    print_info(
        "Versão:",
        faiss_version
    )

    print_success(
        "FAISS carregado com sucesso."
    )

except Exception as error:

    raise RuntimeError(
        "Erro ao carregar a biblioteca FAISS."
    ) from error

print("=" * 70)


 VERIFICAÇÃO DA BIBLIOTECA FAISS
Biblioteca:               FAISS
Versão:                   1.15.0

✅ FAISS carregado com sucesso.


In [ ]:
# Monta o Google Drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


Configuração do Projeto

In [ ]:
# Define os caminhos utilizados no Notebook 03

PROJECT_PATH = Path(
    "/content/drive/MyDrive/RAG_Novo_Embeddings"
)

EMBEDDINGS_DIR = (
    PROJECT_PATH
    / "02_Embeddings"
)

FAISS_DIR = (
    PROJECT_PATH
    / "03_FAISS"
)

EMBEDDINGS_FILE = (
    EMBEDDINGS_DIR
    / "embeddings.npy"
)

METADATA_FILE = (
    EMBEDDINGS_DIR
    / "metadata.pkl"
)

EMBEDDINGS_MANIFEST_FILE = (
    EMBEDDINGS_DIR
    / "manifesto_embeddings.csv"
)

FAISS_INDEX_FILE = (
    FAISS_DIR
    / "faiss.index"
)

FAISS_METADATA_FILE = (
    FAISS_DIR
    / "faiss_metadata.json"
)

FAISS_INFO_FILE = (
    FAISS_DIR
    / "index_info.json"
)

print_header(
    "CONFIGURAÇÃO DO PROJETO"
)

print_info(
    "Projeto:",
    PROJECT_PATH
)

print_info(
    "Embeddings:",
    EMBEDDINGS_DIR
)

print_info(
    "FAISS:",
    FAISS_DIR
)

print_info(
    "Embeddings (.npy):",
    EMBEDDINGS_FILE
)

print_info(
    "Metadados (.pkl):",
    METADATA_FILE
)

print_info(
    "Manifesto embeddings:",
    EMBEDDINGS_MANIFEST_FILE
)

print_info(
    "Índice FAISS:",
    FAISS_INDEX_FILE
)

print_info(
    "Metadados FAISS:",
    FAISS_METADATA_FILE
)

print_info(
    "Manifesto FAISS:",
    FAISS_INFO_FILE
)

print_success(
    "Caminhos configurados com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO PROJETO
Projeto:                  /content/drive/MyDrive/RAG_Novo_Embeddings
Embeddings:               /content/drive/MyDrive/RAG_Novo_Embeddings/02_Embeddings
FAISS:                    /content/drive/MyDrive/RAG_Novo_Embeddings/03_FAISS
Embeddings (.npy):        /content/drive/MyDrive/RAG_Novo_Embeddings/02_Embeddings/embeddings.npy
Metadados (.pkl):         /content/drive/MyDrive/RAG_Novo_Embeddings/02_Embeddings/metadata.pkl
Manifesto embeddings:     /content/drive/MyDrive/RAG_Novo_Embeddings/02_Embeddings/manifesto_embeddings.csv
Índice FAISS:             /content/drive/MyDrive/RAG_Novo_Embeddings/03_FAISS/faiss.index
Metadados FAISS:          /content/drive/MyDrive/RAG_Novo_Embeddings/03_FAISS/faiss_metadata.json
Manifesto FAISS:          /content/drive/MyDrive/RAG_Novo_Embeddings/03_FAISS/index_info.json

✅ Caminhos configurados com sucesso.


Verificação dos Artefatos

In [ ]:
# Verifica se todos os artefatos necessários existem

print_header(
    "VERIFICAÇÃO DOS ARTEFATOS"
)

required_files = {

    "Embeddings (.npy)": EMBEDDINGS_FILE,

    "Metadados (.pkl)": METADATA_FILE,

    "Manifesto": EMBEDDINGS_MANIFEST_FILE

}

missing_files = []

for name, path in required_files.items():

    if path.exists():

        print_info(
            f"{name}:",
            "OK"
        )

    else:

        print_error(
            f"{name} não encontrado."
        )

        print_info(
            "Esperado em:",
            path
        )

        missing_files.append(
            str(path)
        )

if missing_files:

    raise FileNotFoundError(
        "Existem artefatos obrigatórios ausentes."
    )

print_success(
    "Todos os artefatos foram localizados."
)

print("=" * 70)


 VERIFICAÇÃO DOS ARTEFATOS
Embeddings (.npy):        OK
Metadados (.pkl):         OK
Manifesto:                OK

✅ Todos os artefatos foram localizados.


Carregamento dos Embeddings e Metadados

In [ ]:
# Carrega os embeddings e os metadados gerados no Notebook 02

print_header(
    "CARREGAMENTO DOS EMBEDDINGS"
)

embeddings_matrix = np.load(
    EMBEDDINGS_FILE
)

with open(
    METADATA_FILE,
    "rb"
) as file:

    metadata_records = pickle.load(
        file
    )

print_info(
    "Embeddings:",
    embeddings_matrix.shape[0]
)

print_info(
    "Dimensão:",
    embeddings_matrix.shape[1]
)

print_info(
    "Metadados:",
    len(
        metadata_records
    )
)

print_success(
    "Embeddings e metadados carregados com sucesso."
)

print("=" * 70)


 CARREGAMENTO DOS EMBEDDINGS
Embeddings:               6844
Dimensão:                 384
Metadados:                6844

✅ Embeddings e metadados carregados com sucesso.


Validação dos Embeddings

In [ ]:
# Valida a consistência dos embeddings carregados

print_header(
    "VALIDAÇÃO DOS EMBEDDINGS"
)

embedding_norms = np.linalg.norm(
    embeddings_matrix,
    axis=1
)

print_info(
    "Quantidade de vetores:",
    embeddings_matrix.shape[0]
)

print_info(
    "Dimensão:",
    embeddings_matrix.shape[1]
)

print_info(
    "Tipo:",
    embeddings_matrix.dtype
)

print_info(
    "Metadados:",
    len(
        metadata_records
    )
)

print_info(
    "Norma mínima:",
    round(
        float(
            embedding_norms.min()
        ),
        6
    )
)

print_info(
    "Norma máxima:",
    round(
        float(
            embedding_norms.max()
        ),
        6
    )
)

print_info(
    "Norma média:",
    round(
        float(
            embedding_norms.mean()
        ),
        6
    )
)

print()

# Conta a origem dos textos usados na geração dos embeddings
source_counts = {
    "cleaned_summary": 0,
    "summary": 0,
    "text": 0
}

invalid_sources = []

for position, record in enumerate(
    metadata_records
):

    embedding_source = record.get(
        "embedding_source"
    )

    if embedding_source in source_counts:

        source_counts[
            embedding_source
        ] += 1

    else:

        invalid_sources.append(
            {
                "position": position,
                "article_name": record.get(
                    "article_name",
                    ""
                ),
                "chunk_id": record.get(
                    "chunk_id"
                ),
                "embedding_source": embedding_source
            }
        )

print_info(
    "Fonte cleaned_summary:",
    source_counts["cleaned_summary"]
)

print_info(
    "Fonte summary:",
    source_counts["summary"]
)

print_info(
    "Fonte text:",
    source_counts["text"]
)

print()

# Valida quantidade de embeddings e metadados
if embeddings_matrix.shape[0] != len(
    metadata_records
):

    raise ValueError(
        "O número de embeddings não corresponde "
        "ao número de metadados."
    )

# Valida dimensão
if embeddings_matrix.shape[1] != 384:

    raise ValueError(
        "A dimensão dos embeddings está incorreta."
    )

# Valida tipo
if embeddings_matrix.dtype != np.float32:

    raise TypeError(
        "Os embeddings devem estar no formato float32."
    )

# Valida normalização
if not np.allclose(
    embedding_norms,
    1.0,
    atol=1e-5
):

    raise ValueError(
        "Existem embeddings que não estão normalizados."
    )

# Valida embedding_source
if invalid_sources:

    print_error(
        f"Foram encontrados {len(invalid_sources)} "
        "registro(s) com embedding_source inválido."
    )

    print()

    for item in invalid_sources[:5]:

        print(
            f"Artigo: {item['article_name']}"
        )

        print(
            f"Chunk: {item['chunk_id']}"
        )

        print(
            f"embedding_source: "
            f"{item['embedding_source']}"
        )

        print(
            "-" * 70
        )

    raise ValueError(
        "Existem metadados com embedding_source inválido."
    )

if sum(
    source_counts.values()
) != len(
    metadata_records
):

    raise ValueError(
        "A contagem das fontes de embedding "
        "não corresponde ao total de metadados."
    )

print_success(
    "Embeddings e metadados validados com sucesso."
)

print("=" * 70)


 VALIDAÇÃO DOS EMBEDDINGS
Quantidade de vetores:    6844
Dimensão:                 384
Tipo:                     float32
Metadados:                6844
Norma mínima:             1.0
Norma máxima:             1.0
Norma média:              1.0

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158


✅ Embeddings e metadados validados com sucesso.


Construção do Índice FAISS

In [ ]:
# Constrói o índice vetorial FAISS

print_header(
    "CONSTRUÇÃO DO ÍNDICE FAISS"
)

FAISS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

embedding_dimension = embeddings_matrix.shape[1]

index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(
    embeddings_matrix
)

print_info(
    "Tipo do índice:",
    "IndexFlatIP"
)

print_info(
    "Dimensão:",
    embedding_dimension
)

print_info(
    "Vetores indexados:",
    index.ntotal
)

print_info(
    "Métrica:",
    "Produto Interno (Cosine Similarity)"
)

if index.ntotal != len(metadata_records):

    raise ValueError(
        "O número de vetores indexados "
        "não corresponde ao número de metadados."
    )

print_success(
    "Índice FAISS construído com sucesso."
)

print("=" * 70)


 CONSTRUÇÃO DO ÍNDICE FAISS
Tipo do índice:           IndexFlatIP
Dimensão:                 384
Vetores indexados:        6844
Métrica:                  Produto Interno (Cosine Similarity)

✅ Índice FAISS construído com sucesso.


Salvamento do Índice e Metadados

In [ ]:
# Salva o índice FAISS, os metadados e as informações do índice

print_header(
    "SALVAMENTO DO ÍNDICE E METADADOS"
)

FAISS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Conta as fontes utilizadas na geração dos embeddings

embedding_source_counts = {
    "cleaned_summary": sum(
        record.get("embedding_source") == "cleaned_summary"
        for record in metadata_records
    ),
    "summary": sum(
        record.get("embedding_source") == "summary"
        for record in metadata_records
    ),
    "text": sum(
        record.get("embedding_source") == "text"
        for record in metadata_records
    )
}

# Valida a contagem das fontes

if sum(
    embedding_source_counts.values()
) != len(
    metadata_records
):

    raise ValueError(
        "A contagem das fontes dos embeddings "
        "não corresponde ao total de metadados."
    )

# Salva o índice FAISS

faiss.write_index(
    index,
    str(FAISS_INDEX_FILE)
)

# Salva os metadados em JSON

with open(
    FAISS_METADATA_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata_records,
        file,
        ensure_ascii=False,
        indent=2
    )

# Cria o manifesto técnico do índice

index_info = {
    "index_type": "IndexFlatIP",
    "metric": "Inner Product / Cosine Similarity",
    "embedding_dimension": int(
        embedding_dimension
    ),
    "total_vectors": int(
        index.ntotal
    ),
    "total_metadata_records": len(
        metadata_records
    ),
    "embedding_model": (
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
    "embedding_source_counts": {
        "cleaned_summary": int(
            embedding_source_counts["cleaned_summary"]
        ),
        "summary": int(
            embedding_source_counts["summary"]
        ),
        "text": int(
            embedding_source_counts["text"]
        )
    },
    "faiss_version": getattr(
        faiss,
        "__version__",
        "unknown"
    ),
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    )
}

with open(
    FAISS_INFO_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        index_info,
        file,
        ensure_ascii=False,
        indent=2
    )

print_info(
    "Índice:",
    FAISS_INDEX_FILE.name
)

print_info(
    "Metadados:",
    FAISS_METADATA_FILE.name
)

print_info(
    "Manifesto:",
    FAISS_INFO_FILE.name
)

print_info(
    "Vetores salvos:",
    index.ntotal
)

print()

print_info(
    "Fonte cleaned_summary:",
    embedding_source_counts["cleaned_summary"]
)

print_info(
    "Fonte summary:",
    embedding_source_counts["summary"]
)

print_info(
    "Fonte text:",
    embedding_source_counts["text"]
)

print_success(
    "Índice, metadados e manifesto salvos com sucesso."
)

print("=" * 70)


 SALVAMENTO DO ÍNDICE E METADADOS
Índice:                   faiss.index
Metadados:                faiss_metadata.json
Manifesto:                index_info.json
Vetores salvos:           6844

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

✅ Índice, metadados e manifesto salvos com sucesso.


Verificação dos Arquivos Salvos

In [ ]:
# Verifica se os arquivos salvos podem ser carregados corretamente

print_header(
    "VERIFICAÇÃO DOS ARQUIVOS SALVOS"
)

# Reabre o índice FAISS
loaded_index = faiss.read_index(
    str(FAISS_INDEX_FILE)
)

# Reabre os metadados
with open(
    FAISS_METADATA_FILE,
    "r",
    encoding="utf-8"
) as file:

    loaded_metadata = json.load(
        file
    )

# Reabre o manifesto do índice
with open(
    FAISS_INFO_FILE,
    "r",
    encoding="utf-8"
) as file:

    loaded_info = json.load(
        file
    )

print_info(
    "Vetores no índice:",
    loaded_index.ntotal
)

print_info(
    "Metadados:",
    len(
        loaded_metadata
    )
)

print_info(
    "Tipo do índice:",
    loaded_info["index_type"]
)

print_info(
    "Dimensão:",
    loaded_info["embedding_dimension"]
)

if loaded_index.ntotal != len(loaded_metadata):

    raise ValueError(
        "O índice FAISS e os metadados possuem "
        "quantidades diferentes."
    )

if loaded_info["embedding_dimension"] != 384:

    raise ValueError(
        "A dimensão registrada no manifesto "
        "está incorreta."
    )

print_success(
    "Arquivos verificados com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO DOS ARQUIVOS SALVOS
Vetores no índice:        6844
Metadados:                6844
Tipo do índice:           IndexFlatIP
Dimensão:                 384

✅ Arquivos verificados com sucesso.


Verificação Final do Notebook

In [ ]:
# Realiza a verificação final do Notebook 03

print_header(
    "VERIFICAÇÃO FINAL DO NOTEBOOK 03"
)

generated_files = {
    "Índice FAISS": FAISS_INDEX_FILE,
    "Metadados FAISS": FAISS_METADATA_FILE,
    "Manifesto FAISS": FAISS_INFO_FILE
}

missing_files = []

for name, path in generated_files.items():

    if path.exists():

        print_info(
            f"{name}:",
            "OK"
        )

    else:

        print_info(
            f"{name}:",
            "AUSENTE"
        )

        missing_files.append(
            str(path)
        )

if missing_files:

    raise FileNotFoundError(
        "Existem arquivos finais do Notebook 03 "
        "que não foram encontrados no Google Drive."
    )

print()

print_info(
    "Tipo do índice:",
    loaded_info["index_type"]
)

print_info(
    "Vetores:",
    loaded_index.ntotal
)

print_info(
    "Dimensão:",
    loaded_info["embedding_dimension"]
)

print()

# Recupera as contagens das fontes dos embeddings
loaded_source_counts = loaded_info.get(
    "embedding_source_counts",
    {}
)

print_info(
    "Fonte cleaned_summary:",
    loaded_source_counts.get(
        "cleaned_summary",
        0
    )
)

print_info(
    "Fonte summary:",
    loaded_source_counts.get(
        "summary",
        0
    )
)

print_info(
    "Fonte text:",
    loaded_source_counts.get(
        "text",
        0
    )
)

print()

print_info(
    "Arquivo índice:",
    FAISS_INDEX_FILE.name
)

print_info(
    "Arquivo metadados:",
    FAISS_METADATA_FILE.name
)

print_info(
    "Manifesto:",
    FAISS_INFO_FILE.name
)

# Verificações finais de consistência

if loaded_index.ntotal != len(
    loaded_metadata
):

    raise ValueError(
        "O número de vetores do índice não corresponde "
        "ao número de metadados carregados."
    )

if loaded_info["embedding_dimension"] != 384:

    raise ValueError(
        "A dimensão registrada no manifesto está incorreta."
    )

if loaded_info["total_vectors"] != loaded_index.ntotal:

    raise ValueError(
        "O total de vetores registrado no manifesto "
        "não corresponde ao índice carregado."
    )

if sum(
    loaded_source_counts.values()
) != loaded_index.ntotal:

    raise ValueError(
        "A contagem das fontes dos embeddings "
        "não corresponde ao total de vetores."
    )

print_success(
    "Notebook 03 configurado e validado com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO FINAL DO NOTEBOOK 03
Índice FAISS:             OK
Metadados FAISS:          OK
Manifesto FAISS:          OK

Tipo do índice:           IndexFlatIP
Vetores:                  6844
Dimensão:                 384

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

Arquivo índice:           faiss.index
Arquivo metadados:        faiss_metadata.json
Manifesto:                index_info.json

✅ Notebook 03 configurado e validado com sucesso.
